### Selector Group Chat
SelectorGroupChat is a group chat similar to RoundRobinGroupChat, but with a model-based next speaker selection mechanism.

In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage

from dotenv import load_dotenv
import os

In [2]:
from azure.keyvault.secrets import SecretClient
from azure.identity import DefaultAzureCredential


load_dotenv()

vault_url = os.getenv("AZURE_VAULT_URL")

credential = DefaultAzureCredential()

client = SecretClient(credential=credential, vault_url=vault_url)

open_api_key = client.get_secret("openai-api-key").value
open_api_version = client.get_secret("openai-api-version").value
openai_deployment = client.get_secret("openai-deployment").value
openai_endpoint = client.get_secret("openai-endpoint").value

In [ ]:
model_client = AzureOpenAIChatCompletionClient(
    model="gpt-4o-2024-11-20",
    azure_deployment=openai_deployment,
    api_key=open_api_key,
    api_version=open_api_version,
    azure_endpoint=openai_endpoint,
)

In [4]:
sys_message = '''
You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are :
        WebSearchAgent : Searches for information.
        DataAnalystAgent : Performs calculations

    You only plan and delegate tasks - you do not exectue them yourself.

    When assigning tasks, use the below format:
    1. <agent> : <task>

    After all the tasks are completed, summarize the findings and end with "TERMINATE"  
'''

In [5]:
planning_agent = AssistantAgent(
    name = 'PlanningAgent',
    description= 'An agent for planning tasks,this agent should be the first to engage when given a new task.',
    model_client=model_client,
    system_message=sys_message,
)

In [6]:
def search_web_tool(query:str) -> str:
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."


web_search_agent = AssistantAgent(
    name = 'WebSearchAgent',
    description= 'An agent for searching the web for information.',
    model_client=model_client,
    tools = [search_web_tool],
    system_message='''
        You are a web search agent.
        Your only tool is search_web_tool - use it to find the information you need.

        You make only one search call at a time.
        
        Once you have the results, you never do calculations or data analysis on them.
    ''',
)

In [7]:
def percentage_change_tool(start:float, end:float) -> float:
    # Calculate percentage change
    if start == 0:
        return 0
    return ((end - start) / start) * 100

data_analyst_agent = AssistantAgent(
    name = 'DataAnalystAgent',
    description= 'An agent for performing calculations and data analysis.',
    model_client=model_client,
    tools= [percentage_change_tool],
    system_message='''
        You are a data analyst agent.
        Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.

        If you have not seen the data, ask for it.

    ''',
)

Termination Conditions

In [8]:
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination

text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=20)
combined_termination = text_mention_termination | max_messages_termination

In [9]:
selector_prompt = '''
Select an agent to perform the task.

{roles}

Current conversation history :
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planning agent has assigned task before other agents start working.
Only Select one agent.
'''

In [10]:
'''
Try not to overload the selector prompt with too much information.


selector_prompt (str, optional) – The prompt template to use for selecting the next speaker. 

Available fields: ‘{roles}’, ‘{participants}’, and ‘{history}’. 
1. {participants} is the names of candidates for selection. The format is [“<name1>”, “<name2>”, …]. 
2. {roles} is a newline-separated list of names and descriptions of the candidate agents. The format for each line is: “<name> : <description>”. 

3. {history} is the conversation history formatted as a double newline separated of names and message content. The format for each message is: “<name> : <message content>”.
'''

'\nTry not to overload the selector prompt with too much information.\n\n\nselector_prompt (str, optional) – The prompt template to use for selecting the next speaker. \n\nAvailable fields: ‘{roles}’, ‘{participants}’, and ‘{history}’. \n1. {participants} is the names of candidates for selection. The format is [“<name1>”, “<name2>”, …]. \n2. {roles} is a newline-separated list of names and descriptions of the candidate agents. The format for each line is: “<name> : <description>”. \n\n3. {history} is the conversation history formatted as a double newline separated of names and message content. The format for each message is: “<name> : <message content>”.\n'

In [11]:
from autogen_agentchat.teams import SelectorGroupChat

selector_team = SelectorGroupChat(
    participants=[planning_agent,web_search_agent,data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True
)

In [12]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"

In [13]:
from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?


d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\autogen_agentchat\teams\_group_chat\_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)


---------- TextMessage (PlanningAgent) ----------
To address the query, we will need to identify the Miami Heat player with the most points in the 2006-2007 season and calculate the percentage change in their total rebounds between the 2007-2008 and 2008-2009 seasons. The process requires two tasks:

1. Identify the Miami Heat player with the highest points in the 2006-2007 season.
2. Calculate the percentage change in this player's total rebounds between the 2007-2008 and 2008-2009 seasons.

Here's the action plan:

1. **WebSearchAgent**: Search for the Miami Heat player with the highest total points in the 2006-2007 NBA season. Provide details about the player and their statistics.
2. **WebSearchAgent**: Look up the total rebounds for that player in both the 2007-2008 and 2008-2009 NBA seasons.
3. **DataAnalystAgent**: Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons.

Once the tasks are completed, I'll summarize the findings.


d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_PkHSCbXAgHR52EVKVLqedi9N', arguments='{"query":"Miami Heat player with highest points in 2006-2007 NBA season"}', name='search_web_tool')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_PkHSCbXAgHR52EVKVLqedi9N', is_error=False)]
---------- ToolCallSummaryMessage (WebSearchAgent) ----------
Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_MmKOF55Eyi8tm85PJXqGOM8B', arguments='{"query":"

TaskResult(messages=[TextMessage(id='521537d6-465f-4ded-97a5-1cbda2945f3b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 28, 12, 57, 16, 953246, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='72792339-cff5-49a6-a6f6-b539781d4d6e', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=164, completion_tokens=232), metadata={}, created_at=datetime.datetime(2026, 4, 28, 12, 57, 22, 282325, tzinfo=datetime.timezone.utc), content="To address the query, we will need to identify the Miami Heat player with the most points in the 2006-2007 season and calculate the percentage change in their total rebounds between the 2007-2008 and 2008-2009 seasons. The process requires two tasks:\n\n1. Identify the Miami Heat player with the highest poin

Custom Selector Function

In [14]:
from typing import Sequence
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage

def selector_func(messages : Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    if(messages[-1].source != planning_agent.name):
        return planning_agent.name
    return None

In [15]:
await selector_team.reset()
selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
    selector_func=selector_func)

In [16]:
from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To determine the Miami Heat player with the highest point total in the 2006-2007 season and calculate the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons:

### Tasks:
1. **WebSearchAgent**: Identify the Miami Heat player who scored the most points during the 2006-2007 NBA season, including total points scored.
2. **WebSearchAgent**: Gather rebound statistics for this player for both the 2007-2008 and 2008-2009 NBA seasons (total rebounds for each season).
3. **DataAnalystAgent**: Calculate the percentage change in the player's total rebounds between the 2007-2008 and 2008-2009 seasons.

Let’s proceed systematically.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[Function

TaskResult(messages=[TextMessage(id='d97383bd-ea0f-4cd3-a908-37e43736c373', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 28, 12, 58, 50, 207831, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='e05bbd68-6a7a-4a6c-b6ef-e43c5edfa7e8', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=164, completion_tokens=163), metadata={}, created_at=datetime.datetime(2026, 4, 28, 12, 58, 58, 898442, tzinfo=datetime.timezone.utc), content="To determine the Miami Heat player with the highest point total in the 2006-2007 season and calculate the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons:\n\n### Tasks:\n1. **WebSearchAgent**: Identify the Miami Heat player who scored the most points during the 2006-2007 N